# SLSL Landmark Inspection Notebook
**Track B - Phase 3: Cleaning & Trajectory Verification**

Compares raw MediaPipe extractions (10 FPS) against cleaned, smoothed, and normalized landmarks (30 FPS) for the 10 starter signs.

In [ ]:
import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

raw_dir = Path('../data/landmarks_raw') if Path('../data/landmarks_raw').exists() else Path('data/landmarks_raw')
clean_dir = Path('../data/landmarks_clean') if Path('../data/landmarks_clean').exists() else Path('data/landmarks_clean')

starters = [
    ('BLUE', 'Blue_001'),
    ('TODAY', 'Today_036'),
    ('NO', 'No_003'),
    ('HELLO', 'Hello_049'),
    ('THANK_YOU', 'Thank you_058'),
    ('YES', 'Yes_002'),
    ('MOTHER', 'Mother_027'),
    ('COME', 'Come_030'),
    ('EAT', 'Eat_021'),
    ('GO', 'Go_034'),
]

print(f'Available starter signs: {len(starters)}')

### 1. Starter Set Inspection & Metrics Summary

In [ ]:
print(f'{"GLOSS":12} | {"Raw Frames (10fps)":20} | {"Clean Frames (30fps)":22} | {"Duration (s)":12} | {"Scale":8}')
print('-' * 80)
for gloss, take in starters:
    r_file = raw_dir / gloss / f'{take}.npz'
    c_file = clean_dir / gloss / f'{take}.npz'
    if r_file.exists() and c_file.exists():
        r_data = np.load(r_file)
        c_data = np.load(c_file)
        r_fc = len(r_data['pose_world'])
        c_fc = len(c_data['pose_world'])
        dur = float(c_data.get('duration_sec', (c_fc - 1) / 30.0))
        scale = float(c_data.get('scale_factor', 1.0))
        print(f'{gloss:12} | {r_fc:20} | {c_fc:22} | {dur:10.2f}s | {scale:7.2f}x')

### 2. Trajectory Comparison: Raw vs Cleaned
Select any gloss to inspect its wrist trajectory across X, Y, and Z axes.

In [ ]:
def plot_wrist_comparison(gloss: str, take: str):
    r_data = np.load(raw_dir / gloss / f'{take}.npz')
    c_data = np.load(clean_dir / gloss / f'{take}.npz')
    
    # Right wrist (idx 16) and Left wrist (idx 15)
    r_pw = r_data['pose_world']
    c_pw = c_data['pose_world']
    
    # Select dominant hand based on activity
    r_var_rw = np.var(r_pw[:, 16, :3])
    r_var_lw = np.var(r_pw[:, 15, :3])
    wrist_idx = 16 if r_var_rw >= r_var_lw else 15
    wrist_name = 'Right Wrist' if wrist_idx == 16 else 'Left Wrist'
    
    raw_w = r_pw[:, wrist_idx, :3]
    clean_w = c_pw[:, wrist_idx, :3]
    
    t_raw = np.linspace(0, (len(raw_w) - 1) / 10.0, len(raw_w))
    t_clean = np.linspace(0, (len(clean_w) - 1) / 30.0, len(clean_w))
    
    fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
    axis_labels = ['X (Lateral, m)', 'Y (Vertical, m)', 'Z (Depth, m)']
    
    for i in range(3):
        axes[i].plot(t_raw, raw_w[:, i], 'o--', color='crimson', alpha=0.6, label='Raw (10 FPS)')
        axes[i].plot(t_clean, clean_w[:, i], '-', color='royalblue', linewidth=2, label='Cleaned (30 FPS, Smoothed)')
        axes[i].set_ylabel(axis_labels[i])
        axes[i].grid(True, alpha=0.3)
        if i == 0:
            axes[i].legend(loc='upper right')
            
    axes[2].set_xlabel('Time (seconds)')
    plt.suptitle(f'{gloss} ({take}) — {wrist_name} Trajectory Comparison', fontsize=14)
    plt.tight_layout()
    plt.show()

# Example: inspect HELLO
plot_wrist_comparison('HELLO', 'Hello_049')

### 3. Inspect Hand Landmarks (21 Joints)
Visualizes the 21 finger joints across valid frames.

In [ ]:
def inspect_hand(gloss: str, take: str):
    c_data = np.load(clean_dir / gloss / f'{take}.npz')
    lh = c_data['left_hand_world']
    rh = c_data['right_hand_world']
    hd = c_data['hand_detected']
    
    print(f'=== {gloss} Hand Tracking Status ===')
    print(f'Total frames (30 FPS): {len(hd)}')
    print(f'Left Hand valid frames:  {np.sum(hd[:, 0])}/{len(hd)} ({np.mean(hd[:, 0])*100:.1f}%)')
    print(f'Right Hand valid frames: {np.sum(hd[:, 1])}/{len(hd)} ({np.mean(hd[:, 1])*100:.1f}%)')
    
    active_hand = rh if np.mean(hd[:, 1]) >= np.mean(hd[:, 0]) else lh
    valid_f = np.where(~np.isnan(active_hand[:, 0, 0]))[0]
    if len(valid_f) > 0:
        mid_f = valid_f[len(valid_f) // 2]
        pts = active_hand[mid_f]
        plt.figure(figsize=(6, 6))
        plt.scatter(pts[:, 0], -pts[:, 1], c='purple', s=40, label='Hand Joints')
        plt.title(f'{gloss} ({take}) — Active Hand Shape at Frame {mid_f}')
        plt.xlabel('X (m)')
        plt.ylabel('Y (m)')
        plt.grid(True, alpha=0.3)
        plt.axis('equal')
        plt.show()

inspect_hand('BLUE', 'Blue_001')